# CNN + BiLSTM + Attention — AIRTLab và webcam trên Kaggle

Notebook độc lập, checkpoint tương thích `src/models/cnn_lstm.py` và backend localhost.

## 1. Chạy ngay với Input AIRTLab của bạn

Cell cấu hình đã đặt đúng `DATA_ROOT` bạn cung cấp, `DATA_MODE='airtlab'`.
Giữ `INIT_CHECKPOINT=None` để train từ backbone ImageNet; bật GPU và Internet khi cần tải weights,
sau đó Restart & Run All. Không cần tự tạo `manifest.csv` hoặc dùng ba CSV hành động có sẵn.

Notebook đọc cấu trúc `non-violent/cam1`, `non-violent/cam2`, `violent/cam1`, `violent/cam2`.
Tên video số giống nhau trong cùng lớp được ghép thành một nhóm, ví dụ
`non-violent/cam1/1.mp4` và `non-violent/cam2/1.mp4`.
Các nhóm được chia gần 70/15/15 theo lớp với seed cố định; hai góc của cùng tình huống không bị chia ra.
Đây là chia theo **tình huống**, chưa phải chia độc lập theo người vì không có ID diễn viên/buổi quay.
Nguồn cấu trúc: [AIRTLab chính thức](https://github.com/airtlab/A-Dataset-for-Automatic-Violence-Detection-in-Videos).

Video ngắn hơn khoảng lấy mẫu 2,25 giây sẽ bị loại cùng góc camera còn lại và ghi vào
`excluded_airtlab.csv`. Video hỏng sẽ báo lỗi, không tự thay bằng ảnh đen.
Manifest được ghi trong `/kaggle/working/webcam_cnn_lstm/manifest.csv`; Input Kaggle chỉ được đọc.
Đường dẫn video trong manifest tương đối với `DATA_ROOT`, không phải thư mục Output.

**AIRTLab được giữ nhãn nguồn `external`.** Checkpoint/ngưỡng được chọn bằng validation AIRTLab;
báo cáo test là kết quả AIRTLab, không phải độ chính xác webcam. Model có thể nạp vào web,
nhưng cần test và fine-tune trên webcam thật trước khi kết luận chất lượng triển khai.

## Khi đã có dữ liệu webcam tự quay

Đổi `DATA_MODE='manifest'`, đặt `MANIFEST_PATH` tới CSV tự chuẩn bị.
Trong chế độ này đường dẫn video tương đối với thư mục chứa CSV; `DATA_ROOT` không được dùng.

```csv
path,label,group_id,split,domain
videos/session01_normal.mp4,0,session01,train,webcam
videos/session01_fight.mp4,1,session01,train,webcam
videos/session02_normal.mp4,0,session02,val,webcam
videos/session02_fight.mp4,1,session02,val,webcam
videos/session03_normal.mp4,0,session03,test,webcam
videos/session03_fight.mp4,1,session03,test,webcam
```

Đây chỉ là mẫu định dạng. Cần nhiều người/buổi quay, đủ 2 lớp cho mỗi split webcam.
Gộp các clip cùng người/buổi quay/video nguồn cùng nhóm trước khi chia; không dùng lại test để train.
Có thể thêm dữ liệu ngoài với `domain=external`, nhưng giữ nguyên split của dữ liệu đã dùng trước đó.
Gán nhãn nhất quán trong mỗi video, cắt đoạn có trạng thái khác nhau trước. Video cần FPS đúng thực tế,
dài ít nhất 2,25 giây. Các mẫu bình thường dễ nhầm gồm vẫy tay, ôm, tập thể dục và đùa giỡn.

In [ ]:
from pathlib import Path
import copy, hashlib, json, random, warnings
import cv2
import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)
from tqdm.auto import tqdm

print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
# RWF-2000: cấu trúc train/val/Fight và train/val/NonFight.
DATA_MODE = 'rwf2000'  # RWF-2000
DATA_ROOT = Path('/kaggle/input/datasets/vulamnguyen/rwf2000/RWF-2000')
OUT_DIR = Path('/kaggle/working/webcam_cnn_lstm')
MANIFEST_PATH = OUT_DIR / 'manifest.csv'  # AIRTLab: tự tạo ở Output, không ghi vào Input
# Với DATA_MODE='manifest', đổi MANIFEST_PATH sang file CSV của bạn.
INIT_CHECKPOINT = None  # Chỉ nạp checkpoint tin cậy, chưa train trên val/test đang dùng.
if DATA_MODE not in {'rwf2000', 'airtlab', 'manifest'}:
    raise ValueError('DATA_MODE phải là rwf2000, airtlab hoặc manifest.')
EVAL_DOMAIN = 'external' if DATA_MODE in {'rwf2000', 'airtlab'} else 'webcam'

CFG = {
    'seed': 42,
    'evaluation_domain': EVAL_DOMAIN,
    'data': {
        'raw_dir': 'data/raw',
        'classes': ['non_violence', 'violence'], 'num_frames': 16, 'image_size': 224,
        'sample_interval_ms': 150, 'eval_stride_seconds': 1.1,
        'resize_extra': 32, 'max_send_side': 384, 'jpeg_quality': 70,
    },
    'model': {
        'backbone': 'resnet50', 'pretrained': True, 'freeze_backbone': False,
        'lstm_hidden': 512, 'lstm_layers': 2, 'bidirectional': True,
        'dropout': 0.5, 'use_attention': False, 'num_classes': 2,
    },
    'train': {
        'batch_size': 2, 'epochs': 25, 'warmup_epochs': 3, 'patience': 7,
        'lr': 1e-4, 'backbone_lr': 1e-5, 'weight_decay': 1e-4,
        'num_workers': 2, 'use_amp': True, 'webcam_fraction': 0.7,
        'checkpoint_dir': 'checkpoints', 'log_dir': 'reports/outputs',
    },
    'infer': {'window_frames': 16, 'stride': 8, 'threshold': 0.5,
              'sample_interval_ms': 150},
}

# Chỉ lấy kiến trúc từ checkpoint cũ; giữ cách lấy mẫu webcam mới.
# Đây là fine-tune với optimizer mới, không phải resume một lần train bị ngắt.
initial_state = None
if INIT_CHECKPOINT:
    initial = torch.load(INIT_CHECKPOINT, map_location='cpu', weights_only=False)
    if initial['cfg']['data']['classes'] != CFG['data']['classes']:
        raise ValueError('Checkpoint có thứ tự nhãn khác. Không được nạp trực tiếp.')
    CFG['model'] = copy.deepcopy(initial['cfg']['model'])
    CFG['model'].update(pretrained=False, freeze_backbone=False)
    initial_state = initial['model']
    del initial

# Frontend live.html hiện cố định 16 frame, 150 ms, max side 384 và JPEG quality 0.7.
# Giữ hợp đồng này; thay các số trên phải cập nhật frontend tương ứng.
assert CFG['data']['num_frames'] == CFG['infer']['window_frames'] == 16
assert CFG['data']['sample_interval_ms'] == CFG['infer']['sample_interval_ms'] == 150
assert CFG['model']['num_classes'] == 2
assert 0 < CFG['train']['webcam_fraction'] <= 1
assert CFG['train']['epochs'] > CFG['train']['warmup_epochs'] >= 0
SEQUENCE_LENGTH = CFG['data']['num_frames']
INTERVAL_SECONDS = CFG['data']['sample_interval_ms'] / 1000
CLIP_SPAN_SECONDS = (SEQUENCE_LENGTH - 1) * INTERVAL_SECONDS  # 2.25 s giữa frame đầu/cuối

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG['seed'])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Device:', DEVICE, '| clip span:', CLIP_SPAN_SECONDS, 'seconds')
print('Model:', CFG['model'])
if DEVICE.type == 'cpu':
    warnings.warn('Chưa bật GPU; train đầy đủ trên CPU sẽ rất chậm.')

## 2. Tạo và kiểm tra manifest (chạy toàn bộ cell bên dưới)

AIRTLab: nếu CSV chưa có, cell tự tạo trong Output rồi kiểm tra ngay. Không cần chạy một cell tạo CSV riêng.
Nếu manifest đã tồn tại, giữ nguyên split và kiểm tra; muốn chia lại phải chủ động dùng Output mới.
Chế độ manifest: đọc CSV tự chuẩn bị, không tự tạo dữ liệu webcam.
Lỗi thư mục/video sẽ được báo ngay ở bước tạo, không thay bằng lỗi thiếu manifest chung chung.
Kiểm tra nhãn, nhóm giữa các split và file trùng nội dung. Bản cắt/re-encode cần được nhóm đúng từ đầu.


In [ ]:
def build_airtlab_manifest(data_root, output_csv, seed):
    """Group both cameras of each class/clip ID before stratified splitting."""
    data_root, output_csv = Path(data_root).resolve(), Path(output_csv)
    rows = []
    for class_name, label in [('non-violent', 0), ('violent', 1)]:
        for camera in ('cam1', 'cam2'):
            folder = data_root / class_name / camera
            if not folder.is_dir():
                raise FileNotFoundError(f'Thiếu thư mục AIRTLab: {folder}')
            videos = sorted(p for p in folder.iterdir()
                            if p.is_file() and p.suffix.lower() in {'.mp4', '.avi', '.mov', '.mkv'})
            if not videos:
                raise ValueError(f'Không có video trong {folder}')
            for video in videos:
                if not video.stem.isdecimal():
                    raise ValueError(f'AIRTLab gốc dùng tên số, không tự đoán cặp camera: {video.name}')
                cap = cv2.VideoCapture(str(video))
                try:
                    fps = float(cap.get(cv2.CAP_PROP_FPS))
                    count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                    ok, _ = cap.read()
                    if not cap.isOpened() or not ok or not np.isfinite(fps) or fps <= 0 or count <= 0:
                        raise ValueError(f'Không đọc được video/FPS: {video}. Kiểm tra Input có video thật.')
                finally:
                    cap.release()
                rows.append({'path': video.relative_to(data_root).as_posix(), 'label': label,
                             'group_id': f'airtlab/{class_name}/{int(video.stem)}',
                             'domain': 'external', 'camera': camera, 'last_time': (count - 1) / fps})
    df = pd.DataFrame(rows)
    for group, subset in df.groupby('group_id'):
        if len(subset) != 2 or set(subset.camera) != {'cam1', 'cam2'}:
            raise ValueError(f'Cặp camera thiếu/trùng: {group}')

    # Có clip AIRTLab chỉ dài 2s: loại CẢ NHÓM nếu một góc không đủ 2.25s.
    # Không kéo giãn thời gian hoặc lặp frame để che việc thiếu chuyển động.
    short_groups = set(df.loc[df.last_time + 1e-6 < CLIP_SPAN_SECONDS, 'group_id'])
    excluded = df[df.group_id.isin(short_groups)].copy()
    excluded['reason'] = 'camera_pair_shorter_than_required_clip_span'
    df = df[~df.group_id.isin(short_groups)].copy()
    rng = np.random.default_rng(seed)
    assignments = {}
    for label in (0, 1):
        groups = sorted(df.loc[df.label == label, 'group_id'].unique())
        if len(groups) < 3:
            raise ValueError(f'Lớp {label} cần ít nhất 3 cặp camera hợp lệ để chia train/val/test.')
        groups = list(rng.permutation(groups))
        n_val = max(1, round(len(groups) * 0.15))
        n_test = max(1, round(len(groups) * 0.15))
        n_train = len(groups) - n_val - n_test
        for split, selected in [('train', groups[:n_train]),
                                ('val', groups[n_train:n_train + n_val]),
                                ('test', groups[n_train + n_val:])]:
            assignments.update({group: split for group in selected})
    df['split'] = df.group_id.map(assignments)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    excluded.to_csv(output_csv.parent / 'excluded_airtlab.csv', index=False)
    df[['path', 'label', 'group_id', 'split', 'domain']].to_csv(output_csv, index=False)
    print(f'AIRTLab: {len(df)} video, {df.group_id.nunique()} cặp camera; loại {len(excluded)} video ngắn/cùng cặp.')
    print('Manifest:', output_csv)
    return df


def validate_manifest(csv_path, data_root=None):
    csv_path = Path(csv_path)
    if not csv_path.is_file():
        raise FileNotFoundError(f'Không tìm thấy manifest: {csv_path}. '
                           'Với AIRTLab, chạy trọn cell tạo và kiểm tra manifest; '
                           'với chế độ manifest, kiểm tra đường dẫn CSV đã chuẩn bị.')
    df = pd.read_csv(csv_path, dtype={'path': str, 'group_id': str, 'split': str, 'domain': str})
    required = {'path', 'label', 'group_id', 'split', 'domain'}
    if not required.issubset(df.columns) or df.empty:
        raise ValueError(f'Manifest phải có dữ liệu và các cột {sorted(required)}')
    if df[list(required)].isna().any().any():
        raise ValueError('Manifest có ô trống.')
    for col in ('path', 'group_id', 'split', 'domain'):
        df[col] = df[col].str.strip()
        if df[col].eq('').any():
            raise ValueError(f'Cột {col} có giá trị trống.')
    if not df.label.isin([0, 1]).all():
        raise ValueError('label chỉ nhận 0 hoặc 1.')
    df['label'] = df.label.astype(int)
    if not df.split.isin(['train', 'val', 'test']).all():
        raise ValueError('split chỉ nhận train/val/test.')
    if not df.domain.isin(['webcam', 'external']).all():
        raise ValueError('domain chỉ nhận webcam/external.')
    base = Path(data_root).resolve() if data_root is not None else csv_path.parent.resolve()
    resolved = []
    for value in df.path:
        rel = Path(value)
        path = (base / rel).resolve()
        if rel.is_absolute() or not path.is_relative_to(base):
            raise ValueError(f'path phải tương đối và nằm trong dataset: {value}')
        if not path.is_file():
            raise FileNotFoundError(path)
        resolved.append(str(path))
    df['resolved_path'] = resolved
    if df.resolved_path.duplicated().any():
        raise ValueError('Một file xuất hiện nhiều lần trong manifest.')
    if (df.groupby('group_id').split.nunique() > 1).any():
        raise ValueError('Rò rỉ dữ liệu: cùng group_id xuất hiện ở nhiều split.')
    hashes = []
    for path in tqdm(df.resolved_path, desc='Kiểm tra file trùng'):
        digest = hashlib.sha256()
        with open(path, 'rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        hashes.append(digest.hexdigest())
    df['sha256'] = hashes
    if df.sha256.duplicated().any():
        raise ValueError('Có video trùng nội dung; bỏ bản sao và kiểm tra lại split/nhãn.')
    for split in ('train', 'val', 'test'):
        target = df[(df.split == split) & (df.domain == EVAL_DOMAIN)]
        if set(target.label) != {0, 1}:
            raise ValueError(f'{split} {EVAL_DOMAIN} cần đủ 2 lớp để train/đánh giá đúng mục tiêu.')
        if target.group_id.nunique() < 3:
            warnings.warn(f'{split} {EVAL_DOMAIN} có dưới 3 nhóm; số liệu có thể dao động mạnh.')
    return df

def build_rwf_manifest(data_root, output_csv, seed):
    rows = []
    for split in ('train', 'val'):
        for folder_name, label in (('NonFight', 0), ('Fight', 1)):
            folder = Path(data_root) / split / folder_name
            if not folder.is_dir():
                raise FileNotFoundError(f'Thiếu thư mục RWF-2000: {folder}')
            for video in sorted(p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in {'.mp4', '.avi', '.mov', '.mkv'}):
                rows.append({'path': str(video.relative_to(data_root)).replace('\\', '/'), 'label': label, 'group_id': video.stem, 'source_split': split, 'domain': 'external'})
    df = pd.DataFrame(rows).sample(frac=1, random_state=seed).reset_index(drop=True)
    # Giữ split gốc của RWF; val được dùng làm validation/test nội bộ khi cần.
    df['split'] = df['source_split'].map({'train': 'train', 'val': 'test'})
    val_rows = df[df.split == 'test'].groupby('label', group_keys=False).apply(lambda x: x.sample(frac=0.5, random_state=seed)).index
    df.loc[val_rows, 'split'] = 'val'
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    df[['path', 'label', 'group_id', 'split', 'domain']].to_csv(output_csv, index=False)
    print(f'RWF-2000: {len(df)} video')
    return df


def prepare_manifest():
    # Cùng cell định nghĩa builder và validator: không phụ thuộc cell tạo CSV riêng.
    # Giữ manifest đã tồn tại để không tự thay đổi split khi chạy lại cell.
    if DATA_MODE == 'rwf2000' and not Path(MANIFEST_PATH).is_file():
        print('Đang tạo manifest từ RWF-2000:', DATA_ROOT)
        build_rwf_manifest(DATA_ROOT, MANIFEST_PATH, CFG['seed'])
    if DATA_MODE == 'airtlab' and not Path(MANIFEST_PATH).is_file():
        print('Chưa có manifest; đang tạo từ AIRTLab:', DATA_ROOT)
        build_airtlab_manifest(DATA_ROOT, MANIFEST_PATH, CFG['seed'])
    if DATA_MODE == 'rwf2000':
        print('Đánh giá trên RWF-2000 (external), chưa phải độ chính xác webcam thực tế.')
    elif DATA_MODE == 'airtlab':
        print('Đánh giá trên AIRTLab (external), chưa phải độ chính xác webcam thực tế.')
    return validate_manifest(MANIFEST_PATH, DATA_ROOT if DATA_MODE in {'rwf2000', 'airtlab'} else None)


manifest = prepare_manifest()
print(manifest.groupby(['split', 'domain', 'label']).size().rename('videos').to_string())
manifest.to_csv(OUT_DIR / 'manifest_resolved.csv', index=False)

## 2b. (Tuy chon) Gop nhieu dataset (RWF-2000 + Real Life Violence + SCVD [+ webcam]) thanh 1 manifest

Chi chay cell nay neu muon train MOT LAN voi nhieu nguon du lieu cung luc
(thay vi chi RWF-2000). validate_manifest() o che do DATA_MODE='manifest'
luon coi thu muc chua file manifest.csv la goc va bat buoc moi video phai
nam THAT SU (khong phai symlink) ben trong goc do, nen ham duoi day se COPY
(khong symlink) tung nguon vao chung mot thu muc COMBINED_ROOT truoc khi
ghep manifest.

Ho tro 3 kieu cau truc thu muc dataset (dung voi 3 dataset dang co tren
Kaggle Input cua ban):
- kind='rwf2000': <root>/train/{Fight,NonFight}, <root>/val/{Fight,NonFight}.
- kind='flat': <root>/<ten_lop>/*.mp4 (khong co san train/val/test, tu
  chia). Vi du Real Life Violence Situations Dataset: <root> tro toi thu
  muc "Real Life Violence Dataset" (co 2 thu muc con NonViolence/, Violence/).
- kind='scvd': <root>/{Train,Test}/{Normal,Violence,Weaponized}. Vi du
  Smart-City CCTV Violence Detection Dataset: <root> tro toi thu muc
  SCVD_converted. Weaponized duoc gop chung vao nhan bao luc (label=1) vi
  day cung la hanh vi bao luc, chi co them vu khi.

Lay dung duong dan Input: trong panel Data ben trai cua Kaggle, ru chuot
vao dung thu muc noi tren (RWF-2000, Real Life Violence Dataset,
SCVD_converted) se hien icon copy path, bam vao roi dan vao 3 bien root
trong cell ben duoi -- duong dan mac dinh trong cell la PHONG DOAN theo ten
dataset thuong gap, can kiem tra lai cho dung slug tren may ban.

Video webcam tu quay (neu da co) van gop them qua webcam_input_root, cau
truc webcam_root/non_violence/*.mp4 va webcam_root/violence/*.mp4; de
webcam_input_root = None neu chua co, cell van chay binh thuong voi 3
nguon con lai (tat ca domain='external').

Sau khi chay xong cell nay, quay len cell cau hinh o tren va dat
DATA_MODE = 'manifest', MANIFEST_PATH = COMBINED_ROOT / 'manifest.csv'
truoc khi chay tiep cac cell con lai (khong can sua gi them, luong
manifest co san se doc dung file nay).

In [ ]:
import shutil


def _copy_dataset_once(src, dst):
    src, dst = Path(src), Path(dst)
    if dst.exists() and any(dst.iterdir()):
        print(f'Da co du lieu: {dst} (bo qua copy)')
        return
    print(f'Dang copy {src} -> {dst} ...')
    shutil.copytree(src, dst, dirs_exist_ok=True)


def _read_video_fps_count(video):
    cap = cv2.VideoCapture(str(video))
    try:
        fps = float(cap.get(cv2.CAP_PROP_FPS))
        count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        ok, _ = cap.read()
        if not cap.isOpened() or not ok or not np.isfinite(fps) or fps <= 0 or count <= 0:
            raise ValueError(f'Khong doc duoc video/FPS: {video}. Kiem tra file co video that.')
    finally:
        cap.release()
    return fps, count


def _split_by_group(df, seed, min_groups=3, label_name='lop'):
    """Chia stratified train/val/test theo group_id (tranh 1 video/nguoi xuat hien o nhieu split)."""
    rng = np.random.default_rng(seed)
    assignments = {}
    for label in sorted(df.label.unique()):
        groups = sorted(df.loc[df.label == label, 'group_id'].unique())
        if len(groups) < min_groups:
            raise ValueError(f'{label_name} {label} can it nhat {min_groups} nhom de chia train/val/test.')
        groups = list(rng.permutation(groups))
        n_val = max(1, round(len(groups) * 0.15))
        n_test = max(1, round(len(groups) * 0.15))
        n_train = len(groups) - n_val - n_test
        for split, selected in [('train', groups[:n_train]),
                                 ('val', groups[n_train:n_train + n_val]),
                                 ('test', groups[n_train + n_val:])]:
            assignments.update({g: split for g in selected})
    return df.group_id.map(assignments)


def build_flat_class_manifest(data_root, class_to_label, seed, prefix, domain='external'):
    """Dataset dang thu muc phang: data_root/<ten_lop>/*.mp4, chua co train/val/test.
    Dung cho vi du 'Real Life Violence Dataset/{NonViolence,Violence}'.
    """
    data_root = Path(data_root)
    rows = []
    for class_name, label in class_to_label.items():
        folder = data_root / class_name
        if not folder.is_dir():
            raise FileNotFoundError(f'Thieu thu muc: {folder}')
        videos = sorted(p for p in folder.rglob('*')
                         if p.is_file() and p.suffix.lower() in {'.mp4', '.avi', '.mov', '.mkv'})
        if not videos:
            raise ValueError(f'Khong co video trong {folder}')
        for video in videos:
            rows.append({'path': video.relative_to(data_root).as_posix(), 'label': label,
                         'group_id': f'{prefix}/{video.stem}', 'domain': domain})
    df = pd.DataFrame(rows)
    df['split'] = _split_by_group(df, seed, label_name=f'{prefix} lop')
    print(f'{prefix}: {len(df)} video')
    return df


def build_scvd_manifest(data_root, seed, domain='external'):
    """SCVD_converted/{Train,Test}/{Normal,Violence,Weaponized}.
    Weaponized duoc gop vao label=1 (bao luc co vu khi van la bao luc).
    """
    data_root = Path(data_root)
    class_to_label = {'Normal': 0, 'Violence': 1, 'Weaponized': 1}
    rows = []
    for source_split in ('Train', 'Test'):
        for class_name, label in class_to_label.items():
            folder = data_root / source_split / class_name
            if not folder.is_dir():
                raise FileNotFoundError(f'Thieu thu muc SCVD: {folder}')
            videos = sorted(p for p in folder.rglob('*')
                             if p.is_file() and p.suffix.lower() in {'.mp4', '.avi', '.mov', '.mkv'})
            if not videos:
                raise ValueError(f'Khong co video trong {folder}')
            for video in videos:
                rows.append({'path': video.relative_to(data_root).as_posix(), 'label': label,
                             'group_id': f'scvd/{class_name}/{video.stem}',
                             'source_split': source_split, 'domain': domain})
    df = pd.DataFrame(rows).sample(frac=1, random_state=seed).reset_index(drop=True)
    # Giu Train/Test goc cua SCVD; tach mot nua Test lam val, giong cach lam voi RWF-2000.
    df['split'] = df['source_split'].map({'Train': 'train', 'Test': 'test'})
    val_rows = (df[df.split == 'test'].groupby('label', group_keys=False)
                .apply(lambda x: x.sample(frac=0.5, random_state=seed)).index)
    df.loc[val_rows, 'split'] = 'val'
    print(f'SCVD: {len(df)} video')
    return df


def build_webcam_rows(webcam_root, seed, domain='webcam'):
    """Video webcam tu quay. Cau truc: webcam_root/non_violence/*.mp4, webcam_root/violence/*.mp4."""
    webcam_root = Path(webcam_root)
    rows = []
    for class_name, label in (('non_violence', 0), ('violence', 1)):
        folder = webcam_root / class_name
        if not folder.is_dir():
            raise FileNotFoundError(f'Thieu thu muc webcam: {folder}')
        videos = sorted(p for p in folder.iterdir()
                         if p.is_file() and p.suffix.lower() in {'.mp4', '.avi', '.mov', '.mkv', '.webm'})
        if not videos:
            raise ValueError(f'Khong co video trong {folder}')
        for video in videos:
            fps, count = _read_video_fps_count(video)
            rows.append({'path': video.relative_to(webcam_root).as_posix(), 'label': label,
                         'group_id': f'webcam/{class_name}/{video.stem}', 'domain': domain,
                         'last_time': (count - 1) / fps})
    df = pd.DataFrame(rows)
    # Clip webcam qua ngan (< do dai clip model can) khong the dung de train/danh gia.
    short = df.last_time + 1e-6 < CLIP_SPAN_SECONDS
    if short.any():
        print(f'Bo {int(short.sum())} clip webcam ngan hon {CLIP_SPAN_SECONDS:.2f}s:',
              df.loc[short, 'group_id'].tolist())
    df = df[~short].copy()
    df['split'] = _split_by_group(df, seed, label_name='webcam lop')
    return df


def build_combined_manifest(sources, webcam_input_root, combined_root, seed):
    """sources: list cac dict {'kind': 'rwf2000'|'flat'|'scvd', 'root': ..., **kwargs}.
    'flat' can them 'class_to_label' va 'prefix'. webcam_input_root co the la None.
    Tra ve DataFrame va ghi ra combined_root/manifest.csv.
    """
    combined_root = Path(combined_root)
    combined_root.mkdir(parents=True, exist_ok=True)
    dfs = []
    for i, src in enumerate(sources):
        sub_name = f"src{i}_{src['kind']}"
        sub = combined_root / sub_name
        _copy_dataset_once(src['root'], sub)
        if src['kind'] == 'rwf2000':
            df = build_rwf_manifest(sub, combined_root / f'_{sub_name}.csv', seed)
        elif src['kind'] == 'flat':
            df = build_flat_class_manifest(sub, src['class_to_label'], seed, prefix=src.get('prefix', sub_name))
        elif src['kind'] == 'scvd':
            df = build_scvd_manifest(sub, seed)
        else:
            raise ValueError(f"Khong ho tro kind={src['kind']}")
        df = df.copy()
        df['path'] = f'{sub_name}/' + df['path']
        dfs.append(df[['path', 'label', 'group_id', 'split', 'domain']])
        (combined_root / f'_{sub_name}.csv').unlink(missing_ok=True)

    if webcam_input_root:
        sub = combined_root / 'webcam'
        _copy_dataset_once(webcam_input_root, sub)
        webcam_df = build_webcam_rows(sub, seed)
        webcam_df = webcam_df.copy()
        webcam_df['path'] = 'webcam/' + webcam_df['path']
        dfs.append(webcam_df[['path', 'label', 'group_id', 'split', 'domain']])

    combined = pd.concat(dfs, ignore_index=True)
    output_csv = combined_root / 'manifest.csv'
    combined.to_csv(output_csv, index=False)
    print(combined.groupby(['domain', 'split', 'label']).size().rename('videos').to_string())
    print('Manifest gop nhieu nguon:', output_csv)
    return combined


# Duong dan Input duoi day la PHONG DOAN theo slug thuong gap tren Kaggle --
# kiem tra lai bang icon "copy path" trong panel Data truoc khi chay.
COMBINED_ROOT = Path('/kaggle/working/combined_dataset')
# build_combined_manifest(
#     sources=[
#         {'kind': 'rwf2000', 'root': '/kaggle/input/vulamnguyen/rwf2000/RWF-2000'},
#         {'kind': 'flat', 'root': '/kaggle/input/mohamedmustafa/real-life-violence-situations-dataset/Real Life Violence Dataset',
#          'class_to_label': {'NonViolence': 0, 'Violence': 1}, 'prefix': 'rlvs'},
#         {'kind': 'scvd', 'root': '/kaggle/input/<ten-dataset-scvd-cua-ban>/SCVD/SCVD_converted'},
#     ],
#     webcam_input_root=None,  # doi thanh duong dan Input webcam khi da quay xong
#     combined_root=COMBINED_ROOT,
#     seed=CFG['seed'],
# )

## 3. Lấy frame theo thời gian và kiểm tra ảnh đầu vào

Train: cửa sổ ngẫu nhiên trong video đã gán nhãn, 16 frame cách nhau 150 ms.
Validation/test: duyệt cửa sổ cố định bước 1,1 giây, thêm cửa sổ cuối video.
Đánh giá **theo cửa sổ**, không lấy trung bình toàn video để che mất hành động ngắn.
Video ngắn hơn cửa sổ hoặc frame lỗi sẽ báo lỗi, không tạo ảnh đen thay thế.

Mô phỏng đường truyền webcam bằng resize cạnh dài tối đa 384 và JPEG quality 70
(xấp xỉ browser JPEG 0.7; bộ mã hóa trình duyệt có thể khác).
Sau đó dùng Resize(size + 32), CenterCrop và chuẩn hóa ImageNet giống web.
Augmentation train dùng cùng thông số cho cả clip để giữ chuyển động nhất quán.

In [ ]:
def inspect_video(path):
    cap = cv2.VideoCapture(str(path))
    try:
        if not cap.isOpened():
            raise ValueError(f'Không mở được video: {path}')
        fps = float(cap.get(cv2.CAP_PROP_FPS))
        count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if not np.isfinite(fps) or fps <= 0 or count <= 0:
            raise ValueError(f'FPS/frame count không hợp lệ: {path}')
        last_time = (count - 1) / fps
        if last_time + 1e-6 < CLIP_SPAN_SECONDS:
            raise ValueError(f'Video quá ngắn ({last_time:.2f}s): {path}')
        return {'fps': fps, 'frame_count': count, 'last_time': last_time}
    finally:
        cap.release()

def clip_indices(fps, frame_count, start_seconds):
    times = start_seconds + np.arange(SEQUENCE_LENGTH) * INTERVAL_SECONDS
    idxs = np.rint(times * fps).astype(int)
    if idxs[0] < 0 or times[-1] > (frame_count - 1) / fps + 1e-6:
        raise ValueError('Cửa sổ vượt quá thời lượng video.')
    return np.clip(idxs, 0, frame_count - 1)

def read_clip(path, fps, frame_count, start_seconds):
    cap = cv2.VideoCapture(str(path))
    frames = []
    try:
        for idx in clip_indices(fps, frame_count, start_seconds):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ok, bgr = cap.read()
            if not ok:
                raise ValueError(f'Không decode được frame {idx}: {path}')
            frames.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    finally:
        cap.release()
    return frames

def eval_starts(last_time):
    last_start = max(0.0, last_time - CLIP_SPAN_SECONDS)
    starts = list(np.arange(0.0, last_start + 1e-8, CFG['data']['eval_stride_seconds']))
    if not np.isclose(starts[-1], last_start):
        starts.append(last_start)
    return starts

def webcam_transport(frame):
    h, w = frame.shape[:2]
    scale = min(1.0, CFG['data']['max_send_side'] / max(h, w))
    frame = cv2.resize(frame, (max(2, round(w * scale)), max(2, round(h * scale))))
    ok, encoded = cv2.imencode('.jpg', cv2.cvtColor(frame, cv2.COLOR_RGB2BGR),
                              [cv2.IMWRITE_JPEG_QUALITY, CFG['data']['jpeg_quality']])
    if not ok:
        raise ValueError('Không mã hóa được JPEG.')
    return cv2.cvtColor(cv2.imdecode(encoded, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)

def to_tensor_clip(frames, train=False, simulate_transport=True):
    size = CFG['data']['image_size']
    resize = transforms.Resize(size + CFG['data']['resize_extra'])
    crop = transforms.CenterCrop(size)
    normalize = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
    flip = train and random.random() < 0.5
    brightness = random.uniform(0.8, 1.2) if train else 1.0
    contrast = random.uniform(0.8, 1.2) if train else 1.0
    result = []
    for frame in frames:
        img = Image.fromarray(webcam_transport(frame) if simulate_transport else frame)
        img = crop(resize(img))
        if flip:
            img = transforms.functional.hflip(img)
        if train:
            img = transforms.functional.adjust_brightness(img, brightness)
            img = transforms.functional.adjust_contrast(img, contrast)
        result.append(normalize(transforms.functional.to_tensor(img)))
    return torch.stack(result)

def audit_videos(df):
    records = []
    for row in tqdm(df.to_dict('records'), desc='Kiểm tra video'):
        info = inspect_video(row['resolved_path'])
        # Kiểm tra cả đầu và cuối; lỗi tại cửa sổ khác cũng sẽ dừng DataLoader khi đọc.
        for start in (0.0, max(0.0, info['last_time'] - CLIP_SPAN_SECONDS)):
            read_clip(row['resolved_path'], info['fps'], info['frame_count'], start)
        records.append({**row, **info})
    return pd.DataFrame(records)

manifest = audit_videos(manifest)
manifest.to_csv(OUT_DIR / 'manifest_resolved.csv', index=False)

# Xem đúng crop mô hình nhìn thấy, nhất là người nằm gần mép webcam.
example = manifest[(manifest.split == 'train') & (manifest.domain == EVAL_DOMAIN)].iloc[0]
preview = to_tensor_clip(read_clip(example.resolved_path, example.fps, example.frame_count, 0.0))
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
for ax, idx in zip(axes, (0, 5, 10, 15)):
    ax.imshow((preview[idx] * std + mean).clamp(0, 1).permute(1, 2, 0).numpy())
    ax.set_title(f'{idx * INTERVAL_SECONDS:.2f}s')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
class WebcamClipDataset(Dataset):
    def __init__(self, df, train=False):
        self.train = train
        self.items = []
        for row in df.to_dict('records'):
            starts = [None] if train else eval_starts(row['last_time'])
            self.items.extend({**row, 'start_seconds': start} for start in starts)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        row = self.items[idx]
        start = (random.uniform(0, max(0.0, row['last_time'] - CLIP_SPAN_SECONDS))
                 if self.train else row['start_seconds'])
        frames = read_clip(row['resolved_path'], row['fps'], row['frame_count'], start)
        return {'frames': to_tensor_clip(frames, self.train), 'label': row['label'],
                'path': row['path'], 'domain': row['domain'], 'group_id': row['group_id'],
                'start_seconds': float(start)}

def sampling_weights(df):
    # Cân bằng lớp trong mỗi domain và ưu tiên webcam; chỉ dùng thống kê TRAIN.
    # Không thêm class-weight loss cùng lúc để tránh bù mất cân bằng hai lần.
    active = set(df.domain)
    fraction = CFG['train']['webcam_fraction']
    mass = {'webcam': fraction, 'external': 1 - fraction}
    if len(active) == 1:
        mass = {next(iter(active)): 1.0}
    counts = df.groupby(['domain', 'label']).size().to_dict()
    nclasses = df.groupby('domain').label.nunique().to_dict()
    return [mass[r.domain] / nclasses[r.domain] / counts[(r.domain, r.label)]
            for r in df.itertuples()]

def seed_worker(worker_id):
    seed = torch.initial_seed() % 2**32
    np.random.seed(seed)
    random.seed(seed)

train_df = manifest[manifest.split == 'train'].reset_index(drop=True)
train_ds = WebcamClipDataset(train_df, train=True)
generator = torch.Generator().manual_seed(CFG['seed'])
sampler = WeightedRandomSampler(sampling_weights(train_df), len(train_ds), replacement=True,
                                generator=generator)
loader_options = dict(batch_size=CFG['train']['batch_size'], num_workers=CFG['train']['num_workers'],
                      pin_memory=DEVICE.type == 'cuda', worker_init_fn=seed_worker)
train_loader = DataLoader(train_ds, sampler=sampler, generator=generator, **loader_options)
val_loader = DataLoader(WebcamClipDataset(manifest[manifest.split == 'val']), **loader_options)
test_loader = DataLoader(WebcamClipDataset(manifest[manifest.split == 'test']), **loader_options)
print('Train videos / validation windows / test windows:',
      len(train_ds), len(val_loader.dataset), len(test_loader.dataset))

## 4. CNN + BiLSTM + Attention

Các lớp bên dưới được đồng bộ từ source hiện tại để giữ tên và kích thước trọng số.
Mặc định ResNet50; có thể chọn ResNet18 khi train mới để giảm chi phí CPU localhost.
Nếu nạp checkpoint cũ, kiến trúc được lấy từ checkpoint và kiểm tra bằng `strict=True`.

In [ ]:
from __future__ import annotations

import torch
import torch.nn as nn
from torchvision.models import ResNet18_Weights, ResNet50_Weights, resnet18, resnet50

_BACKBONES = {
    "resnet18": (resnet18, ResNet18_Weights.IMAGENET1K_V1, 512),
    "resnet50": (resnet50, ResNet50_Weights.IMAGENET1K_V2, 2048),
}


class SpatialEncoder(nn.Module):
    """2D CNN extracts appearance (spatial) features from each frame."""

    def __init__(self, pretrained: bool = True, freeze: bool = False, backbone: str = "resnet50"):
        super().__init__()
        name = backbone.lower()
        if name not in _BACKBONES:
            raise ValueError(f"Unsupported backbone '{backbone}'. Use: {sorted(_BACKBONES)}")
        builder, weights_enum, out_dim = _BACKBONES[name]
        weights = weights_enum if pretrained else None
        net = builder(weights=weights)
        self.features = nn.Sequential(*list(net.children())[:-1])
        self.out_dim = out_dim
        if freeze:
            for p in self.features.parameters():
                p.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B*T, 3, H, W) -> (B*T, 2048)
        return self.features(x).flatten(1)


class TemporalAttention(nn.Module):
    """Learns which frames in the clip contribute most to the decision."""

    def __init__(self, hidden_dim: int):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, h: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        energy = self.score(h).squeeze(-1)
        alpha = torch.softmax(energy, dim=1)
        context = torch.bmm(alpha.unsqueeze(1), h).squeeze(1)
        return context, alpha


class TemporalHead(nn.Module):
    """BiLSTM + attention over a sequence of spatial feature vectors."""

    def __init__(self, input_dim: int, cfg: dict):
        super().__init__()
        m = cfg["model"]
        hidden = m["lstm_hidden"]
        bidirectional = m.get("bidirectional", True)
        n_layers = m.get("lstm_layers", 2)
        dropout = m.get("dropout", 0.5)
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        lstm_out = hidden * (2 if bidirectional else 1)
        self.use_attention = m.get("use_attention", True)
        self.attn = TemporalAttention(lstm_out) if self.use_attention else None
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_out, m.get("num_classes", 2)),
        )

    def forward(self, seq: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor | None]:
        # seq: (B, T, D)
        lstm_out, _ = self.lstm(seq)
        if self.attn is not None:
            pooled, alpha = self.attn(lstm_out)
        else:
            pooled = lstm_out[:, -1]
            alpha = None
        return self.classifier(pooled), alpha


class CNNLSTM(nn.Module):
    """
    Spatial-temporal violence detector.

    Spatial: shared ResNet50 on each frame.
    Temporal: BiLSTM over the frame feature sequence, then attention pooling.
    """

    def __init__(self, cfg: dict):
        super().__init__()
        m = cfg["model"]
        self.spatial = SpatialEncoder(
            pretrained=m.get("pretrained", True),
            freeze=m.get("freeze_backbone", False),
            backbone=m.get("backbone", "resnet50"),
        )
        self.temporal = TemporalHead(self.spatial.out_dim, cfg)

    def encode_spatial(self, frames: torch.Tensor) -> torch.Tensor:
        b, t, c, h, w = frames.shape
        feat = self.spatial(frames.reshape(b * t, c, h, w))
        return feat.view(b, t, -1)

    def forward(self, frames: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor | None]:
        return self.temporal(self.encode_spatial(frames))

In [ ]:
model = CNNLSTM(CFG).to(DEVICE)
if initial_state is not None:
    model.load_state_dict(initial_state, strict=True)
    del initial_state
    initial_state = None

def set_train_stage(model, epoch):
    model.train()
    # Giữ BatchNorm CNN ở eval vì batch nhỏ. Chỉ mở layer4 sau warmup.
    model.spatial.eval()
    for parameter in model.spatial.parameters():
        parameter.requires_grad = False
    if epoch > CFG['train']['warmup_epochs']:
        for parameter in model.spatial.features[7].parameters():
            parameter.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW([
    {'params': model.temporal.parameters(), 'lr': CFG['train']['lr']},
    {'params': model.spatial.features[7].parameters(), 'lr': CFG['train']['backbone_lr']},
], weight_decay=CFG['train']['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['train']['epochs'])
AMP_ENABLED = DEVICE.type == 'cuda' and CFG['train']['use_amp']
scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
print('Parameters:', sum(p.numel() for p in model.parameters()))

## 5. Huấn luyện và chọn checkpoint

Warmup LSTM/attention, sau đó fine-tune tầng cuối CNN bằng learning rate nhỏ.
AMP giảm bộ nhớ GPU, gradient clipping hạn chế gradient lớn.
Chọn checkpoint bằng macro-F1 validation tại ngưỡng 0.5 trên `EVAL_DOMAIN`:
`external` cho AIRTLab, `webcam` cho chế độ manifest webcam. Không dùng test để chọn model.

In [ ]:
def binary_metrics(y, probabilities, threshold):
    y = np.asarray(y, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    pred = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        'accuracy': float(accuracy_score(y, pred)),
        'precision': float(precision_score(y, pred, zero_division=0)),
        'recall': float(recall_score(y, pred, zero_division=0)),
        'f1': float(f1_score(y, pred, zero_division=0)),
        'macro_f1': float(f1_score(y, pred, labels=[0, 1], average='macro', zero_division=0)),
        'roc_auc': float(roc_auc_score(y, probabilities)) if len(set(y)) == 2 else None,
        'false_positive_rate': float(fp / (fp + tn)) if fp + tn else None,
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp), 'n_windows': len(y),
    }

def train_epoch(epoch):
    set_train_stage(model, epoch)
    total, count = 0.0, 0
    for batch in tqdm(train_loader, desc=f'Train {epoch}', leave=False):
        x, y = batch['frames'].to(DEVICE), batch['label'].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
            logits, _ = model(x)
            loss = criterion(logits, y)
        if not torch.isfinite(loss):
            raise RuntimeError('Loss không hữu hạn; kiểm tra dữ liệu/learning rate.')
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total += float(loss.detach()) * len(y)
        count += len(y)
    return total / count

@torch.no_grad()
def collect_predictions(loader):
    model.eval()
    rows = []
    for batch in tqdm(loader, desc='Evaluate', leave=False):
        # FP32 giống inference web; tránh sai lệch ngưỡng vì AMP ở validation.
        logits, _ = model(batch['frames'].to(DEVICE))
        probabilities = logits.softmax(1)[:, 1].cpu().tolist()
        for i, probability in enumerate(probabilities):
            rows.append({'path': batch['path'][i], 'label': int(batch['label'][i]),
                         'domain': batch['domain'][i], 'group_id': batch['group_id'][i],
                         'start_seconds': float(batch['start_seconds'][i]),
                         'violence_prob': probability})
    return pd.DataFrame(rows)

history = []
best_score, stale = -1.0, 0
best_path = OUT_DIR / 'best.pt'
for epoch in range(1, CFG['train']['epochs'] + 1):
    loss = train_epoch(epoch)
    val_predictions = collect_predictions(val_loader)
    target_val = val_predictions[val_predictions.domain == EVAL_DOMAIN]
    metrics = binary_metrics(target_val.label, target_val.violence_prob, 0.5)
    history.append({'epoch': epoch, 'train_loss': loss, **metrics})
    print(f"epoch={epoch:02d} loss={loss:.4f} {EVAL_DOMAIN}_val_macro_f1={metrics['macro_f1']:.4f}")
    scheduler.step()
    if metrics['macro_f1'] > best_score + 1e-6:
        best_score, stale = metrics['macro_f1'], 0
        export_cfg = copy.deepcopy(CFG)
        export_cfg['model']['pretrained'] = False  # Đã có weights, localhost không cần tải ImageNet.
        torch.save({'model': {k: v.detach().cpu() for k, v in model.state_dict().items()},
                    'cfg': export_cfg, 'val': metrics, 'epoch': epoch,
                    'training_protocol': f'group-held-out {EVAL_DOMAIN}, fixed-time windows'}, best_path)
    else:
        stale += 1
    (OUT_DIR / 'history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')
    if stale >= CFG['train']['patience']:
        print('Early stopping')
        break

pd.DataFrame(history).plot(x='epoch', y=['train_loss', 'macro_f1'], subplots=True, figsize=(8, 6))
plt.tight_layout()
plt.savefig(OUT_DIR / 'training_curves.png')
plt.show()

## 6. Chọn ngưỡng từ validation, rồi đánh giá test

Tối đa macro-F1 trên validation của `EVAL_DOMAIN`, báo cáo precision, recall và tỷ lệ báo nhầm.
Ngưỡng AIRTLab chưa được hiệu chỉnh trên webcam thật. Không dùng test để chỉnh ngưỡng.
Các cửa sổ chồng lấn không phải các mẫu độc lập. Đây chưa phải đo cảnh báo/phút hoặc độ trễ camera.

In [ ]:
checkpoint = torch.load(best_path, map_location='cpu', weights_only=False)
model.load_state_dict(checkpoint['model'], strict=True)
val_predictions = collect_predictions(val_loader)
target_val = val_predictions[val_predictions.domain == EVAL_DOMAIN]

def choose_threshold(predictions):
    candidates = []
    for threshold in np.linspace(0.05, 0.95, 91):
        metrics = binary_metrics(predictions.label, predictions.violence_prob, float(threshold))
        candidates.append({'threshold': float(threshold), **metrics})
    # Hòa điểm: chọn gần 0.5 nhất, không dựa vào test.
    best = max(candidates, key=lambda row: (row['macro_f1'], -abs(row['threshold'] - 0.5)))
    return best, pd.DataFrame(candidates)

selected, threshold_table = choose_threshold(target_val)
threshold = selected['threshold']
threshold_table.to_csv(OUT_DIR / 'validation_thresholds.csv', index=False)
checkpoint['cfg']['infer']['threshold'] = threshold
checkpoint['val_tuned'] = selected
torch.save(checkpoint, best_path)
CFG['infer']['threshold'] = threshold
print(f'Ngưỡng chọn từ validation {EVAL_DOMAIN}:', threshold)

test_predictions = collect_predictions(test_loader)  # Chỉ sau khi đã khóa mô hình và ngưỡng.
reports = {}
for split, predictions in [('val', val_predictions), ('test', test_predictions)]:
    predictions['prediction'] = (predictions.violence_prob >= threshold).astype(int)
    predictions.to_csv(OUT_DIR / f'{split}_window_predictions.csv', index=False)
    for domain in sorted(predictions.domain.unique()):
        subset = predictions[predictions.domain == domain]
        reports[f'{split}_{domain}'] = {
            'threshold_0.5': binary_metrics(subset.label, subset.violence_prob, 0.5),
            'tuned_threshold': binary_metrics(subset.label, subset.violence_prob, threshold),
            'n_videos': int(subset.path.nunique()), 'n_groups': int(subset.group_id.nunique()),
        }

target_test = test_predictions[test_predictions.domain == EVAL_DOMAIN]
print(classification_report(target_test.label, target_test.prediction, labels=[0, 1],
                            target_names=CFG['data']['classes'], zero_division=0))
print(json.dumps(reports[f'test_{EVAL_DOMAIN}'], indent=2))
(OUT_DIR / 'metrics.json').write_text(json.dumps(reports, indent=2), encoding='utf-8')
cm = confusion_matrix(target_test.label, target_test.prediction, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap='Blues')
ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=CFG['data']['classes'],
       yticklabels=CFG['data']['classes'], xlabel='Predicted', ylabel='True', title=f'{EVAL_DOMAIN} test windows')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center')
plt.tight_layout()
plt.savefig(OUT_DIR / f'{EVAL_DOMAIN}_test_confusion.png')
plt.show()

In [ ]:
# Kiểm tra load checkpoint bằng một model mới, không tải ImageNet.
deployment_model = CNNLSTM(checkpoint['cfg']).eval()
deployment_model.load_state_dict(checkpoint['model'], strict=True)
del deployment_model
(OUT_DIR / 'deployment_config.json').write_text(
    json.dumps(checkpoint['cfg'], indent=2), encoding='utf-8')
print('Checkpoint:', best_path, '| MB:', round(best_path.stat().st_size / 1e6, 1))
print('Xuất kèm metrics.json, history.json, manifest_resolved.csv và *_window_predictions.csv')

## 7. Đưa checkpoint về localhost

1. Tải `webcam_cnn_lstm/best.pt` và báo cáo từ Output Kaggle. Giữ một bản checkpoint cũ để so sánh.
2. Chép checkpoint mới vào `checkpoints/best.pt` trong project và **restart** web:

   ```powershell
   python -m uvicorn src.api:app --host 127.0.0.1 --port 8000
   ```

3. Xem `/health`, sau đó mở `/live`. Backend đọc cấu hình và ngưỡng từ checkpoint mới.
   Không cần sửa ngưỡng trong `default.yaml`. Ngưỡng được chọn trên domain ghi trong checkpoint; AIRTLab chưa thay thế việc hiệu chỉnh webcam thật.
4. Frontend hiện lấy 16 frame cách khoảng 150 ms, giảm cạnh dài còn 384 và JPEG quality 0.7;
   notebook mô phỏng thông số này. Chỉ đánh giá sau khi buffer đủ 16 frame (khoảng 2,4 giây).
   Frontend cũ cho phép request từ 8 frame; những dự đoán khởi động đó chưa khớp luồng train.
   Thời gian lấy frame trình duyệt có thể trễ; cần đo lại trên máy sử dụng thật.
5. Thử các buổi quay mới, ghi nhận báo nhầm và bỏ sót. Không dùng lại test này để train rồi tiếp tục
   báo cáo nó như test độc lập. Muốn fine-tune tiếp phải giữ một test khác chưa sử dụng.

**Phạm vi tương thích:** checkpoint nạp được vào backend hiện tại; kiến trúc, RGB, resize/crop và
chuẩn hóa giống source. Notebook này thay đổi train/evaluation, không thay source web.
`src.infer` ở đường **upload** vẫn dùng cửa sổ frame liên tiếp; công cụ đánh giá local cũ đã được gỡ.
Đường upload chưa tái hiện cách lấy mẫu theo thời gian của notebook. Không so trực tiếp các số liệu đó
với `metrics.json`; dùng notebook để đánh giá video webcam theo đúng protocol này.

Khung người HOG trên web không phải định vị người đang bạo lực. Phần trăm là xác suất cho cả clip.
Nếu kết quả vẫn kém, xem các cửa sổ dự đoán sai trong CSV, kiểm tra crop/FPS/nhãn và bổ sung dữ liệu
webcam tương ứng trước khi tăng kích thước mô hình.